# Phase 3 — Portfolio Construction

**Purpose:** Demonstrate the `Constructor` chain with all constraint types.
Shows `ConstructionReport` output including per-constraint results and relaxation notes.
Uses the synthetic market fixture; no live data required.

In [1]:
# Cell 1: Imports and setup
import pathlib
import sys
from datetime import date

import pandas as pd

import ah_research
from ah_research.backtest.types import Signals
from ah_research.portfolio.constructor import Constraint, ConstructionReport, Constructor

sys.path.insert(0, str(pathlib.Path().resolve() / "tests"))
from fixtures.phase2.synthetic_market import build_synthetic_market

code_version = getattr(ah_research, "__version__", "dev")
print(f"Python : {sys.version.split()[0]}")
print(f"ah-research: {code_version}")

Python : 3.12.10
ah-research: 0.0.1


In [2]:
# Cell 2: Build synthetic market and signals
ASOF = date(2024, 6, 30)
SYMBOLS = [f"60000{i}.SH" for i in range(10)]

repo = build_synthetic_market(
    start=date(2024, 1, 1),
    end=ASOF,
    symbols=SYMBOLS,
)

# Build synthetic signal: ascending rank scores
signals_df = pd.DataFrame(
    {
        "date": pd.to_datetime([str(ASOF)] * len(SYMBOLS)),
        "symbol": SYMBOLS,
        "signal": [float(i + 1) for i in range(len(SYMBOLS))],  # 1..10
    }
)
signals = Signals.from_dataframe(signals_df)

print(f"Symbols  : {len(SYMBOLS)}")
print(f"Signals  :\n{signals_df.to_string(index=False)}")

Symbols  : 10
Signals  :
      date    symbol  signal
2024-06-30 600000.SH     1.0
2024-06-30 600001.SH     2.0
2024-06-30 600002.SH     3.0
2024-06-30 600003.SH     4.0
2024-06-30 600004.SH     5.0
2024-06-30 600005.SH     6.0
2024-06-30 600006.SH     7.0
2024-06-30 600007.SH     8.0
2024-06-30 600008.SH     9.0
2024-06-30 600009.SH    10.0


In [3]:
# Cell 3: Basic constructor — top 20%, equal weight, no constraints
report_basic: ConstructionReport = (
    Constructor(signals, repo=repo, asof=ASOF)
    .method("top_quantile", quantile=0.2)
    .weight_by("equal")
    .build()
)

print("=== Basic (top 20%, equal weight) ===")
print(f"Method used         : {report_basic.method_used}")
print(f"Weighting scheme    : {report_basic.weighting_scheme}")
print(f"Final positions     : {report_basic.final_position_count}")
print()
print("Weights:")
print(report_basic.weights.to_string(index=False))

=== Basic (top 20%, equal weight) ===
Method used         : top_quantile
Weighting scheme    : equal
Final positions     : 2

Weights:
   symbol  weight
600008.SH     0.5
600009.SH     0.5


In [4]:
# Cell 4: Constructor with max_weight constraint
report_capped: ConstructionReport = (
    Constructor(signals, repo=repo, asof=ASOF)
    .method("top_quantile", quantile=0.4)
    .weight_by("signal_proportional")
    .constrain(Constraint.max_weight(0.30))
    .build()
)

print("=== With max_weight(0.30) constraint ===")
print(f"Final positions     : {report_capped.final_position_count}")
print(f"Max weight          : {report_capped.weights['weight'].max():.4f}  (cap=0.30)")
print(f"Weights sum to 1    : {report_capped.weights['weight'].sum():.6f}")
print()
print("Constraint results:")
for cr in report_capped.constraint_results:
    print(f"  kind={cr.constraint.kind!r}  status={cr.status!r}  detail={cr.detail!r}")

=== With max_weight(0.30) constraint ===
Final positions     : 3
Max weight          : 0.3333  (cap=0.30)
Weights sum to 1    : 1.000000

Constraint results:
  kind='max_weight'  status='bound'  detail=''


In [5]:
# Cell 5: Multiple constraints — min/max positions + max_weight
report_multi: ConstructionReport = (
    Constructor(signals, repo=repo, asof=ASOF)
    .method("top_n", n=5)
    .weight_by("equal")
    .constrain(
        Constraint.max_weight(0.25),
        Constraint.min_positions(3),
        Constraint.max_positions(8),
    )
    .build()
)

print("=== Multi-constraint (top_n=5, max_w=0.25, min_pos=3, max_pos=8) ===")
print(f"Final positions     : {report_multi.final_position_count}")
print(f"Weights sum         : {report_multi.weights['weight'].sum():.6f}")
print()
print("Constraint results:")
for cr in report_multi.constraint_results:
    print(f"  kind={cr.constraint.kind!r}  status={cr.status!r}  detail={cr.detail!r}")
if report_multi.relaxation_notes:
    print()
    print("Relaxation notes:")
    for note in report_multi.relaxation_notes:
        print(f"  - {note}")

=== Multi-constraint (top_n=5, max_w=0.25, min_pos=3, max_pos=8) ===
Final positions     : 5
Weights sum         : 1.000000

Constraint results:
  kind='min_positions'  status='slack'  detail=''
  kind='max_positions'  status='slack'  detail=''
  kind='max_weight'  status='slack'  detail=''


In [6]:
# Cell 6: All constraint types demo
print("=== Available Constraint Factories ===")
constraints = [
    Constraint.max_weight(0.05),
    Constraint.max_gross(0.50),
    Constraint.sector_neutral_to("CSI300"),
    Constraint.tracking_error(300),
    Constraint.min_positions(10),
    Constraint.max_positions(50),
]
for c in constraints:
    print(f"  kind={c.kind!r:30s}  params={c.params}  priority={c.priority}")

=== Available Constraint Factories ===
  kind='max_weight'                    params={'w': 0.05}  priority=50
  kind='max_gross'                     params={'gross': 0.5}  priority=40
  kind='sector_neutral_to'             params={'benchmark': 'CSI300'}  priority=60
  kind='tracking_error'                params={'bps': 300}  priority=70
  kind='min_positions'                 params={'n': 10}  priority=10
  kind='max_positions'                 params={'n': 50}  priority=20


In [7]:
# Cell 7: Reproducibility block
import subprocess

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

print("=== Reproducibility ===")
print(f"Code version : {code_version}")
print(f"Git SHA      : {git_sha}")
print(f"Symbols      : {SYMBOLS}")
print(f"As-of        : {ASOF}")
print("Data source  : synthetic (build_synthetic_market)")

=== Reproducibility ===
Code version : 0.0.1
Git SHA      : d786f88
Symbols      : ['600000.SH', '600001.SH', '600002.SH', '600003.SH', '600004.SH', '600005.SH', '600006.SH', '600007.SH', '600008.SH', '600009.SH']
As-of        : 2024-06-30
Data source  : synthetic (build_synthetic_market)


---
> **DISCLAIMER:** Results are based on synthetic / randomly generated data. This is a portfolio construction demonstration only and does not constitute investment advice. Past performance does not guarantee future results.